# Contrôle de cohérence — Source 4 : MongoDB (events)

Objectif : vérifier la volumétrie, la présence effective des anomalies volontaires, et la flexibilité du schéma, avant de considérer la source comme prête pour l'intégration.

In [ ]:
import os
import pandas as pd
from pymongo import MongoClient

try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

MONGO_URI = os.getenv("MONGO_URI", "mongodb://localhost:27017")
DB_NAME = os.getenv("DB_NAME", "edusmart_mongo")
COLLECTION_NAME = os.getenv("COLLECTION_NAME", "events")

client = MongoClient(MONGO_URI)
collection = client[DB_NAME][COLLECTION_NAME]

# On accumule les résultats ici pour le tableau de synthèse final
resultats = []

## 1. Volumétrie globale

In [ ]:
duplicate_count = NB_EVENTS // 200

total = collection.count_documents({})
print(f"Nombre total de documents : {total}")

# calculés dans generate_data.py). Affiche un message clair si l'écart
# est anormal (ex: > 1% d'écart).
expected_total = NB_EVENTS + duplicate_count
if abs(total - expected_total) > expected_total * 0.01:
    print("Attention : l'écart entre le nombre de documents et la plage attendue est anormal.")
else:
    print("Le nombre de documents est cohérent avec la plage attendue.")
    print(f"Nombre attendu de documents : {expected_total}")
    print(f"Écart : {total - expected_total} documents")

## 2. Vérification des anomalies volontaires

Pour chaque anomalie, compare le taux observé au taux que tu as codé dans `inject_anomalies()` (les seuils `random.random() < X`). Un écart de quelques dixièmes de point est normal (aléatoire), un écart de plusieurs points doit t'alerter.

In [ ]:
# --- 2.1 Événements sans student_code (attendu ~1%) ---
sans_student_code = collection.count_documents({"student_code": {"$exists": False}})  # TODO
taux = sans_student_code / total * 100
print(f"Sans student_code : {sans_student_code} ({taux:.2f}%)")
resultats.append(("student_code manquant", sans_student_code, taux, "~1%"))

In [ ]:
# --- 2.2 Champs absents (device, city, ip_address, etc. — attendu ~2%) ---
# TODO : pour CHAQUE champ de removable_fields dans generate_data.py,
# compte combien de documents ne le possèdent PAS ($exists: False).
# Fais une boucle sur la liste des champs plutôt que de tout répéter à la main.
champs_a_verifier = [
    "device", "operating_system", "app_version", "ip_address",
    "city", "country", "session_id", "duration_seconds", "success",
]
# TODO : boucle + count_documents + affichage + append à resultats
for champ in champs_a_verifier:
    count_absent = collection.count_documents({champ: {"$exists": False}})
    taux_absent = count_absent / total * 100
    print(f"{champ} manquant : {count_absent} ({taux_absent:.2f}%)")
    resultats.append((f"{champ} manquant", count_absent, taux_absent, "~2%"))

In [ ]:
# --- 2.3 Valeurs nulles explicites (attendu ~2.5%) ---
# TODO : pour les mêmes champs, compte cette fois {"champ": None}
# (différent de $exists: False — ici le champ EXISTE mais vaut null).

In [ ]:
# --- 2.4 Villes non canoniques (variantes d'écriture) ---
# TODO : utilise collection.distinct("city") pour lister toutes les valeurs
# distinctes de city, puis compare visuellement à ta liste CITIES_SN propre.
# Combien de variantes "sales" par ville canonique retrouves-tu ?

In [ ]:
# --- 2.5 Versions d'application incohérentes ---
# TODO : collection.distinct("app_version") — vérifie que tu retrouves
# bien les 5 variantes de APP_VERSIONS.

In [ ]:
# --- 2.6 OS écrits différemment ---
# TODO : collection.distinct("operating_system")

In [ ]:
# --- 2.7 Formats de date multiples ---
# TODO : utilise une agrégation avec $group sur { "$type": "$timestamp" }
# pour compter combien de documents ont un timestamp de type "date" (BSON)
# vs "string". Tu dois retrouver une grosse majorité en "date" et une
# petite fraction en "string" (celles touchées par inject_anomalies).
pipeline = [
    {"$group": {"_id": {"$type": "$timestamp"}, "count": {"$sum": 1}}}
]
# TODO : list(collection.aggregate(pipeline)) et affiche le résultat

In [ ]:
# --- 2.8 Adresses IP invalides (attendu ~3%) ---
# TODO : utilise une regex MongoDB ($regex) pour matcher un format IPv4
# valide, puis compte le complémentaire (total - valides - $exists:false).

In [ ]:
# --- 2.9 Durées négatives (attendu ~1%) ---
# TODO : collection.count_documents({"duration_seconds": {"$lt": 0}})

In [ ]:
# --- 2.10 Événements dupliqués (même event_id plusieurs fois) ---
# TODO : agrégation $group sur event_id avec $sum: 1, puis $match count > 1.
# Combien d'event_id distincts sont concernés ? Est-ce cohérent avec
# duplicate_count calculé dans generate_data.py (len(events)//200) ?

## 3. Vérification de la flexibilité du schéma

Objectif : démontrer que les documents n'ont pas tous la même structure, et que les champs varient de façon cohérente selon `event_type` (ex: `quiz_code` uniquement pour QUIZ_STARTED/QUIZ_SUBMITTED, `video_quality`/`buffer_time` uniquement pour VIDEO_STARTED).

In [ ]:
# TODO : pour chaque event_type distinct, prends un échantillon de
# documents (collection.find({"event_type": t}).limit(50)) et liste
# l'ensemble des clés rencontrées (set().union(*[doc.keys() for doc in ...])).
# Affiche un tableau event_type -> champs observés.
# Vérifie en particulier :
# - qu'un LOGIN n'a jamais quiz_code
# - qu'un QUIZ_SUBMITTED a bien score/attempt dans metadata ET quiz_code au 1er niveau

## 4. Tableau de synthèse

In [ ]:
# TODO : construis un DataFrame à partir de la liste `resultats`
# accumulée au fil du notebook (anomalie, count, taux observé, taux attendu)
# et affiche-le. C'est ce tableau que tu pourras réutiliser tel quel
# dans ton document de présentation (étape 5).
df_synthese = pd.DataFrame(resultats, columns=["Anomalie", "Count", "Taux observé (%)", "Taux attendu"])
df_synthese